# RUNTIME: local PC, NVIDIA RTX 3060 GPU (the human's order of 2026-11-19; plan section 10)

Project: `qpain/paper-04` - Stage: `03 update curves` - Colab units: 0

Applies every update method of analysis_plan.md section 5 to the frozen repetitions of notebook 01, with the source
settings chosen by notebook 02, and saves the prediction for every evaluation tablet.
Order of work, fixed by the plan:
1. pseudo-transfer inside the Copenhagen source scales (0 -> 1 and 1 -> 0) chooses the update settings of
   Scenario II; the choice is written to a file before any scale-code-2 tablet is read by a model;
2. Scenario II (scale codes 0+1 -> 2) with the chosen settings;
3. Scenario I (instrument 1 -> 2 and 2 -> 1) with all three values of every update setting.
Evaluation tablets enter prediction and scoring only. Update tablets and evaluation tablets of a repetition are
asserted to be disjoint, and both are asserted to be new-condition tablets outside every source set.

Run: `jupyter nbconvert --to notebook --execute` with `PAPER_ROOT` set; `P4_SMOKE=1` for the smoke test (two full
repetitions per part; it also gives the timing for the cost rule of plan section 7).
Source: `notebooks/src/03_update_curves_GPU.py` (edit the source, rebuild with scripts/py-to-ipynb.py).

In [ ]:
# Setup
import os, sys, json, time, shutil, platform, datetime, subprocess, collections
import importlib.metadata as ilmd
os.environ["HF_HUB_OFFLINE"] = "1"                   # the foundation model's weights are a local, verified file
ROOT = os.environ.get("PAPER_ROOT", os.path.abspath(".."))
SMOKE = os.environ.get("P4_SMOKE", "0") == "1"       # smoke test: two repetitions per part, full settings
SMOKE_TEST = SMOKE
OUT_REAL, CACHE, CKPT = (os.path.join(ROOT, d) for d in ("outputs", "data/cache", "checkpoints"))
# The smoke test never writes to outputs/ or to the real parts, and always starts from nothing
OUT = os.path.join(ROOT, "logs", "smoke-03-out") if SMOKE else OUT_REAL
PARTS = os.path.join(CKPT, "p4_curves_parts_smoke" if SMOKE else "p4_curves_parts")
if SMOKE:
    for d in (OUT, PARTS):
        shutil.rmtree(d, ignore_errors=True)
for d in (OUT, PARTS):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(ROOT, "notebooks"))
NOTEBOOK = "03_update_curves_GPU"

import numpy as np, pandas as pd, torch
import p4_common as P, p4_models as M, p4_updates as U

SEED = 20261122
CNN_SEEDS = [0, 1, 2]
SPLITS_SHA_AUDITED = "034d226f695684bfd80be90445b2d9ba6b6def65ae71660c49acb3bc4dfa7a35"   # notebook 01, run local-run-01
# Design size. The defaults are the plan's (200 repetitions; 3 orders per choice of pool groups). They are lowered
# only by the cost rule of plan section 7, recorded in section 10 first; the parts already computed stay valid.
S1_REPS = int(os.environ.get("P4_S1_REPS", "200"))
S2_ORDERS = int(os.environ.get("P4_S2_ORDERS", "3"))
assert (S1_REPS, S2_ORDERS) in ((200, 3), (100, 2), (100, 1)), "only the three designs named by the cost rule are allowed"
SMOKE_REPS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "no CUDA device: this notebook runs on the GPU in both modes, so that the smoke test times the real run"
torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
VERSIONS = {m: ilmd.version(m) for m in ("numpy", "pandas", "scipy", "scikit-learn", "xgboost", "torch", "tabpfn")}
VERSIONS.update(python=platform.python_version(), device=DEVICE, torch_cuda=str(torch.version.cuda), cudnn=str(torch.backends.cudnn.version()),
                p4_common=P.COMMON_VERSION, p4_models=M.MODELS_VERSION, p4_updates=U.UPDATES_VERSION, gpu=torch.cuda.get_device_name(0))
smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], capture_output=True, text=True)
VERSIONS["gpu_driver"] = smi.stdout.strip() if smi.returncode == 0 else "unknown"
# GPU load before this notebook does any work: above a few percent, another job shares the GPU and the timings are inflated
smi = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"], capture_output=True, text=True)
GPU_BEFORE = smi.stdout.strip() if smi.returncode == 0 else "unknown"      # not part of the run fingerprint
print("Run at", datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), "| SMOKE", SMOKE, "| S1_REPS", S1_REPS, "| S2_ORDERS", S2_ORDERS)
print(VERSIONS)
T0 = time.time()

In [ ]:
# Inputs: arrays, frozen splits, and notebook 02's outputs. Each is checked against its hash before use.
with open(os.path.join(OUT_REAL, "p4_splits.json"), "rb") as f:
    split_bytes = f.read()
SPLITS_SHA = P.sha256_bytes(split_bytes)
assert SPLITS_SHA == SPLITS_SHA_AUDITED, "p4_splits.json is not the file written and audited by notebook 01"
SPLITS = json.loads(split_bytes)
with np.load(os.path.join(CACHE, "p4_arrays.npz")) as npz:
    A = {k: npz[k] for k in npz.files}
assert P.array_digest(A) == SPLITS["arrays_sha256"], "the cached arrays are not the ones the frozen splits index"
with open(os.path.join(OUT_REAL, "p4_complete_02.json"), encoding="utf-8") as f:
    MARK02 = json.load(f)
assert MARK02["notebook"] == "02_tune_source_GPU" and MARK02["smoke"] is False, "notebook 02 has not finished a full run"
for name, sha in MARK02["files"].items():
    assert P.sha256_file(os.path.join(OUT_REAL, name)) == sha, f"{name} differs from notebook 02's completion marker"
with open(os.path.join(OUT_REAL, "p4_tuning_chosen.json"), encoding="utf-8") as f:
    TUNED = json.load(f)
assert TUNED["fingerprint"] == MARK02["fingerprint"] and TUNED["splits_sha256"] == SPLITS_SHA and TUNED["arrays_sha256"] == SPLITS["arrays_sha256"]
assert TUNED["seed"] == SEED and TUNED["cnn_seeds"] == CNN_SEEDS and TUNED["tabpfn"]["sha256"] == M.TABPFN_SHA256
CHOSEN = TUNED["chosen"]
WEIGHTS = M.tabpfn_weights(os.path.join(CKPT, "tabpfn"), download=False)

GRIDS_ALL = dict(pls_copies=list(U.COPIES_GRID), xgb_copies=list(U.COPIES_GRID), cnn_steps=list(U.CNN_STEPS_GRID),
                 tabpfn_repeat=list(U.CONTEXT_REPEAT_GRID), t_factor=list(U.LAMBDA_FACTOR_GRID))
code_hashes = {f: P.sha256_file(os.path.join(ROOT, "notebooks", f)) for f in ("src/03_update_curves_GPU.py", "p4_models.py", "p4_common.py", "p4_updates.py")}
# Run fingerprint of the parts. The design size (S1_REPS, S2_ORDERS) is left out on purpose: it selects which
# repetitions are run and assembled, not how one repetition is computed.
FINGERPRINT = P.sha256_bytes(json.dumps(dict(
    code=code_hashes, splits=SPLITS_SHA, arrays=SPLITS["arrays_sha256"], tuned=MARK02["files"]["p4_tuning_chosen.json"], seed=SEED,
    cnn_seeds=CNN_SEEDS, smoke=SMOKE, versions=VERSIONS, weights=M.TABPFN_SHA256, grids=GRIDS_ALL, ft_lr=U.CNN_FT_LR,
    u1_min=U.U1_MIN_FOR_SLOPE), sort_keys=True).encode("utf-8"))
print("splits, arrays, notebook 02 outputs and weights verified | run fingerprint", FINGERPRINT[:16])

In [ ]:
# Source models and the update machinery
FAMILY_LABELS = ["pls", "xgb", "tabpfn"] + [f"cnn{s}" for s in CNN_SEEDS]       # the CNN is kept per seed
CKPT_BY = {(r["source_set"], r["seed"]): r for r in TUNED["cnn_checkpoints"]}


def feat(x_raw, source_set, family, dataset):
    """Features of one family: the preprocessing chosen for that family on that source set (per spectrum, nothing fitted)."""
    x = M.preprocess(x_raw, CHOSEN[source_set][family]["preprocessing"])
    return M.bin_pairs(x) if (family == "tabpfn" and dataset == "shootout") else x


def build_source(source_set, x_raw, y, dataset):
    """Source models of the four families for one source set, with the settings chosen by notebook 02."""
    c = CHOSEN[source_set]
    src = dict(name=source_set, dataset=dataset, y=np.asarray(y, dtype=np.float64),
               x={fam: feat(x_raw, source_set, fam, dataset) for fam in M.FAMILIES})
    src["k"] = int(c["pls"]["setting"]["n_components"])
    src["pls"] = M.pls_fit(src["x"]["pls"], src["y"], src["k"])
    src["b"], src["b0"] = U.pls_linear(src["pls"], src["x"]["pls"].shape[1])
    src["xgb_params"], src["xgb_trees"] = c["xgb"]["setting"], int(c["xgb"]["setting"]["n_trees"])
    src["xgb"] = M.xgb_fit(src["x"]["xgb"], src["y"], src["xgb_params"], src["xgb_trees"], SEED)
    src["cnn"] = []
    for s in CNN_SEEDS:
        row = CKPT_BY[(source_set, s)]
        path = os.path.join(ROOT, row["file"])
        assert P.sha256_file(path) == row["sha256"], f"{path} is not the checkpoint notebook 02 recorded"
        model, scaler = U.load_cnn(path, DEVICE)
        ref = M.Scaler(src["x"]["cnn"], src["y"])              # the statistics the checkpoint must hold, recomputed from the source rows
        assert scaler.x_mean.shape == ref.x_mean.shape and np.allclose(scaler.x_mean, ref.x_mean, rtol=1e-6, atol=1e-9) and all(
            abs(a - b) <= 1e-6 * abs(b) + 1e-9 for a, b in ((scaler.x_sd, ref.x_sd), (scaler.y_mean, ref.y_mean), (scaler.y_sd, ref.y_sd))), \
            f"{path}: the stored scaling does not match the source rows"
        fitted = M.rmse(src["y"], M.cnn_predict(model, scaler, src["x"]["cnn"], DEVICE))
        assert abs(fitted - row["rmse_on_own_training_tablets"]) <= 1e-3 * row["rmse_on_own_training_tablets"], \
            f"{path}: the reloaded CNN does not reproduce its recorded error on its training tablets ({fitted} vs {row['rmse_on_own_training_tablets']})"
        src["cnn"].append((model, scaler))
    return src


def predict_source(src, x_raw):
    """Predictions of every source model (no update) for raw spectra: {family label: vector}."""
    f = {fam: feat(x_raw, src["name"], fam, src["dataset"]) for fam in M.FAMILIES}
    out = dict(pls=M.pls_predict(src["pls"], f["pls"]), xgb=np.asarray(src["xgb"].predict(f["xgb"]), dtype=np.float64),
               tabpfn=M.tabpfn_predict(src["x"]["tabpfn"], src["y"], f["tabpfn"], WEIGHTS, SEED, DEVICE))
    for s, (model, scaler) in zip(CNN_SEEDS, src["cnn"]):
        out[f"cnn{s}"] = M.cnn_predict(model, scaler, f["cnn"], DEVICE)
    for k, v in out.items():
        M.require_finite(v, f"source predictions {k}")
    return out, f


def new_condition(src, rows, x_raw_all, y_all):
    """New-condition tablets of one part: global row numbers, features per family, reference values, and every source
    model's prediction with no update (U0). `pos` maps a global row number to its position here."""
    rows = np.asarray(rows)
    base, f = predict_source(src, x_raw_all[rows])
    return dict(rows=rows, pos={int(r): i for i, r in enumerate(rows)}, x=f, y=np.asarray(y_all[rows], dtype=np.float64), base=base)


def update_once(src, new, upd, ev, grids, tag, secs):
    """All update methods for one update set. upd, ev: positions into the new-condition part (disjoint).
    Returns {label: predictions for ev}; label = family|update|setting."""
    assert len(set(upd.tolist())) == len(upd) and not (set(upd.tolist()) & set(ev.tolist())), "update and evaluation tablets overlap"
    out, y_u = {}, new["y"][upd]
    t = time.time()
    for lab in FAMILY_LABELS:                                   # U1: the same rule for every family
        a, b = U.slope_bias(new["base"][lab][upd], y_u)
        out[f"{lab}|U1|-"] = a + b * new["base"][lab][ev]
    f = new["x"]["pls"]
    for w in grids["pls_copies"]:                               # U2, PLS: refit on source plus weighted update tablets
        xs, ys = U.stack_with_copies(src["x"]["pls"], src["y"], f[upd], y_u, w)
        out[f"pls|U2|{w}"] = M.pls_predict(M.pls_fit(xs, ys, src["k"]), f[ev])
    for fac in grids["t_factor"]:                               # T: Tikhonov-type updating of the PLS vector
        b, b0, _ = U.tikhonov(src["b"], f[upd], y_u, fac)
        out[f"pls|T|{fac:g}"] = f[ev] @ b + b0
    secs["pls_u1_t"] += time.time() - t
    t = time.time()
    f = new["x"]["xgb"]
    for w in grids["xgb_copies"]:                               # U2, boosted trees: refit, settings unchanged
        xs, ys = U.stack_with_copies(src["x"]["xgb"], src["y"], f[upd], y_u, w)
        out[f"xgb|U2|{w}"] = np.asarray(M.xgb_fit(xs, ys, src["xgb_params"], src["xgb_trees"], SEED).predict(f[ev]), dtype=np.float64)
    secs["xgb"] += time.time() - t
    t = time.time()
    f = new["x"]["cnn"]
    for s, (model, scaler) in zip(CNN_SEEDS, src["cnn"]):       # U2, CNN: fine-tuning, one trajectory per seed
        snaps = U.cnn_finetune(model, scaler, src["x"]["cnn"], src["y"], f[upd], y_u, f[ev], grids["cnn_steps"], U.CNN_FT_LR,
                               U.seed_for(*tag, "cnn", s), DEVICE)
        for st in grids["cnn_steps"]:
            out[f"cnn{s}|U2|{st}"] = snaps[st]
    secs["cnn"] += time.time() - t
    t = time.time()
    f = new["x"]["tabpfn"]
    for w in grids["tabpfn_repeat"]:                            # U2, foundation model: context = source rows, then update tablets
        xs, ys = U.stack_with_copies(src["x"]["tabpfn"], src["y"], f[upd], y_u, w)
        out[f"tabpfn|U2|{w}"] = M.tabpfn_predict(xs, ys, f[ev], WEIGHTS, SEED, DEVICE)
    secs["tabpfn"] += time.time() - t
    for k, v in out.items():
        M.require_finite(v, f"predictions {k} {tag}")
        assert v.shape == (len(ev),)
    return out


def part_path(unit, name):
    os.makedirs(os.path.join(PARTS, unit), exist_ok=True)
    return os.path.join(PARTS, unit, name + ".npz")


def save_part(path, preds, meta):
    labels = sorted(preds)
    tmp = path[:-4] + ".tmp.npz"
    np.savez(tmp, labels=np.array(labels), pred=np.stack([preds[k] for k in labels]).astype(np.float32),
             meta=np.array(json.dumps(dict(meta, fingerprint=FINGERPRINT), sort_keys=True)))
    os.replace(tmp, path)


def expected_labels(grids, sizes, with_u0):
    """Every key a part must hold: family|update|setting|n."""
    per_n = [f"{lab}|U1|-" for lab in FAMILY_LABELS] + [f"pls|U2|{w}" for w in grids["pls_copies"]] + [f"pls|T|{f:g}" for f in grids["t_factor"]] \
        + [f"xgb|U2|{w}" for w in grids["xgb_copies"]] + [f"cnn{s}|U2|{st}" for s in CNN_SEEDS for st in grids["cnn_steps"]] \
        + [f"tabpfn|U2|{w}" for w in grids["tabpfn_repeat"]]
    out = [f"{k}|{n}" for n in sizes for k in per_n] + ([f"{lab}|U0|-|0" for lab in FAMILY_LABELS] if with_u0 else [])
    assert len(out) == len(set(out))
    return sorted(out)


def load_part(path, expect, labels_expected):
    """Predictions as stored (32-bit), so that a resumed run and an uninterrupted one give the same numbers."""
    with np.load(path) as z:
        meta = json.loads(str(z["meta"]))
        labels, pred = z["labels"].tolist(), z["pred"]
    assert labels == labels_expected, f"{path}: the methods stored are not the expected ones"
    assert pred.dtype == np.float32 and pred.shape == (len(labels), len(expect["eval"])) and np.isfinite(pred).all(), f"{path}: bad prediction block"
    preds = dict(zip(labels, pred))
    assert meta["fingerprint"] == FINGERPRINT, f"{path} was written by another run (code, settings, inputs, packages): delete the parts folder"
    for k, v in expect.items():
        assert meta[k] == v, f"{path}: {k} is {meta[k]}, expected {v}"
    return preds, meta


def run_part(unit, name, src, new, upd_by_n, ev_rows, grids, with_u0, ident):
    """One part file: every update method for each update size in upd_by_n ({n: global rows}), evaluated on ev_rows.
    Keys of the result: family|update|setting|n. Reuses the part if it exists under this run fingerprint."""
    path = part_path(unit, name)
    ev = np.array([new["pos"][int(r)] for r in ev_rows])
    expect = dict(unit=unit, ident=ident, eval=[int(r) for r in ev_rows], updates={str(n): [int(r) for r in rows] for n, rows in upd_by_n.items()},
                  grids=grids, with_u0=with_u0)
    if not os.path.exists(path):
        secs, preds, t = collections.defaultdict(float), {}, time.time()
        if with_u0:
            for lab in FAMILY_LABELS:
                preds[f"{lab}|U0|-|0"] = new["base"][lab][ev]
        for n, rows in upd_by_n.items():
            assert len(rows) == n
            upd = np.array([new["pos"][int(r)] for r in rows])
            for k, v in update_once(src, new, upd, ev, grids, (unit, name, n), secs).items():
                preds[f"{k}|{n}"] = v
        save_part(path, preds, dict(expect, seconds=dict(secs), seconds_total=time.time() - t))
    return load_part(path, expect, expected_labels(grids, list(upd_by_n), with_u0))


def score_rows(unit, rep, preds, y):
    rows = []
    for key, p in preds.items():
        lab, update, setting, n = key.split("|")
        fam, seed = (("cnn", int(lab[3:])) if lab.startswith("cnn") else (lab, -1))
        err = p.astype(np.float64) - y
        rows.append(dict(unit=unit, rep=rep, family=fam, seed=seed, update=update, setting=setting, n=int(n),
                         rmse=float(np.sqrt(np.mean(err ** 2))), bias=float(np.mean(err)), n_eval=int(len(y))))
    return rows


UNITS, SUMMARY, SECONDS = {}, [], []


def assemble(unit, rep_ids, pred_list, eval_list, y_all, labels_expected, extra=None):
    """Stack the repetitions of one part of the design: pred (repetition, label, evaluation tablet), 32-bit."""
    labels = sorted(pred_list[0])
    assert labels == sorted(labels_expected), f"{unit}: the methods assembled are not the expected ones"
    assert all(sorted(p) == labels for p in pred_list), f"{unit}: repetitions do not hold the same methods"
    assert len(rep_ids) == len(pred_list) == len(eval_list) == len(set(rep_ids)) and len(set(len(e) for e in eval_list)) == 1
    ev = np.array(eval_list)
    UNITS[unit] = dict(**(extra(ev) if extra else {}), labels=np.array(labels), pred=np.stack([np.stack([p[k] for k in labels]) for p in pred_list]).astype(np.float32),
                       eval_rows=ev.astype(np.int32), y_true=np.asarray(y_all, dtype=np.float64)[ev], rep=np.array(rep_ids, dtype=np.int32))
    for r, p, e in zip(rep_ids, pred_list, eval_list):
        SUMMARY.extend(score_rows(unit, r, p, np.asarray(y_all, dtype=np.float64)[np.array(e)]))


c_x, c_y, c_scale, c_group = A["c_x"], A["c_content"], A["c_scale"], A["c_group"]
S1, S2, PT = SPLITS["scenario_I"], SPLITS["scenario_II"], SPLITS["pseudo_transfer"]
NEW_SCALE = S2["new_scale"]


def copenhagen_extra(train_rows):
    """Companion columns for a Copenhagen prediction file, so that notebook 04 needs no other file for the group and range tables."""
    lo, hi = float(c_y[train_rows].min()), float(c_y[train_rows].max())
    return lambda ev: dict(eval_group=c_group[ev], eval_type=A["c_type"][ev].astype(np.int32), source_content_min=np.array(lo), source_content_max=np.array(hi))


def group_updates(rep, per_group):
    """Update tablets of a group-structured repetition: the first `per_group` tablets of each pool group, pool groups in their listed order."""
    return [int(r) for g in rep["pool_groups"] for r in rep["pool_order"][g][:per_group]]

In [ ]:
# 1. Pseudo-transfer inside the source scales: chooses the update settings of Scenario II (plan section 5).
# No scale-code-2 tablet is read by any model in this cell (asserted on every row set).
PSEUDO = (("pseudo_01", "scale0_to_scale1", "pseudo_scale0", 0, 1), ("pseudo_10", "scale1_to_scale0", "pseudo_scale1", 1, 0))
assert PT["per_group"] == 2
for unit, key, source_set, s_from, s_to in PSEUDO:
    train = np.array(PT[key]["train"])
    new_rows = np.flatnonzero(c_scale == s_to)
    assert set(c_scale[train].tolist()) == {s_from} and NEW_SCALE not in (s_from, s_to)
    src = build_source(source_set, c_x[train], c_y[train], "copenhagen")
    new = new_condition(src, new_rows, c_x, c_y)
    reps = PT[key]["repetitions"][:SMOKE_REPS] if SMOKE else PT[key]["repetitions"]
    pl, el, ids = [], [], []
    for rep in reps:
        upd = group_updates(rep, PT["per_group"])
        assert set(upd) | set(rep["eval"]) <= set(new_rows.tolist()) and not (set(upd) & set(rep["eval"]))
        assert not (set(c_group[upd].tolist()) & set(c_group[rep["eval"]].tolist())), "an evaluation group is in the update pool"
        preds, meta = run_part(unit, f"rep{rep['rep']:03d}", src, new, {len(upd): upd}, rep["eval"], GRIDS_ALL, True, rep["rep"])
        pl.append(preds); el.append(rep["eval"]); ids.append(rep["rep"])
        SECONDS.append(dict(unit=unit, part=f"rep{rep['rep']:03d}", **{k: round(v, 2) for k, v in meta["seconds"].items()}, total=round(meta["seconds_total"], 2)))
    assemble(unit, ids, pl, el, c_y, expected_labels(GRIDS_ALL, [8], True), copenhagen_extra(train))
    print(unit, "| repetitions", len(ids), "| seconds per repetition", round(np.mean([s["total"] for s in SECONDS if s["unit"] == unit]), 1), flush=True)

# The choice: per update method, mean RMSE at n = 8 per direction (CNN: seeds averaged per repetition first), the two
# directions averaged with equal weight, lowest wins, tie to the middle value and then to the smaller.
ps = pd.DataFrame([r for r in SUMMARY if r["unit"] in ("pseudo_01", "pseudo_10") and r["update"] in ("U2", "T")])
assert set(ps["n"]) == {8}
ps = ps.groupby(["unit", "rep", "family", "update", "setting"], as_index=False)["rmse"].mean()        # seeds averaged
pt_table = ps.groupby(["unit", "family", "update", "setting"], as_index=False).agg(mean_rmse=("rmse", "mean"), repetitions=("rmse", "size"))
METHOD_GRID = {("pls", "U2"): ("pls_copies", U.COPIES_GRID), ("xgb", "U2"): ("xgb_copies", U.COPIES_GRID), ("cnn", "U2"): ("cnn_steps", U.CNN_STEPS_GRID),
               ("tabpfn", "U2"): ("tabpfn_repeat", U.CONTEXT_REPEAT_GRID), ("pls", "T"): ("t_factor", U.LAMBDA_FACTOR_GRID)}
choice, GRIDS_S2 = {}, {}
for (fam, upd_code), (gname, grid) in METHOD_GRID.items():
    sub = pt_table[(pt_table["family"] == fam) & (pt_table["update"] == upd_code)]
    table = [[float(sub[(sub["unit"] == u) & (sub["setting"] == f"{v:g}")]["mean_rmse"].iloc[0]) for v in grid] for u in ("pseudo_01", "pseudo_10")]
    value, avg = U.choose_setting(grid, table)
    choice[gname] = dict(family=fam, update=upd_code, grid=list(grid), chosen=value, mean_rmse_scale0_to_1=table[0], mean_rmse_scale1_to_0=table[1],
                         average=[float(a) for a in avg])
    GRIDS_S2[gname] = [value]
settings_doc = dict(rule="plan section 5: mean RMSE at n = 8 per direction, directions averaged with equal weight, lowest wins, tie to the middle value then the smaller",
                    fingerprint=FINGERPRINT, splits_sha256=SPLITS_SHA, tuned_sha256=MARK02["files"]["p4_tuning_chosen.json"], smoke=SMOKE,
                    repetitions={u: int(len(UNITS[u]["rep"])) for u in ("pseudo_01", "pseudo_10")}, scenario_II=choice,
                    scenario_I=dict(rule="no pseudo-transfer exists; the middle value is the main setting and all three are reported",
                                    main={g: grid[U.MIDDLE] for g, grid in GRIDS_ALL.items()}))
SETTINGS_TEXT = json.dumps(settings_doc, sort_keys=True, indent=1)
SETTINGS_PATH = os.path.join(OUT, "p4_update_settings.json")
if os.path.exists(SETTINGS_PATH):                    # a resumed run must arrive at the same choice
    with open(SETTINGS_PATH, encoding="utf-8") as f:
        assert f.read() == SETTINGS_TEXT, "the pseudo-transfer choice differs from the one already written: stop and investigate"
else:
    with open(SETTINGS_PATH + ".tmp", "w", encoding="utf-8", newline="\n") as f:
        f.write(SETTINGS_TEXT)
    os.replace(SETTINGS_PATH + ".tmp", SETTINGS_PATH)
print("update settings for Scenario II (written before any scale-code-2 tablet is read):", {g: v[0] for g, v in GRIDS_S2.items()})

In [ ]:
# 2. Scenario II: scale codes 0+1 -> 2, held-out tablet groups, with the settings chosen above.
assert os.path.exists(SETTINGS_PATH), "the update settings must be on disk before Scenario II starts"
s2_train = np.array(S2["train"])
s2_new = np.flatnonzero(c_scale == NEW_SCALE)
assert NEW_SCALE not in set(c_scale[s2_train].tolist()) and len(s2_new) == 120 and not (set(s2_train.tolist()) & set(s2_new.tolist()))
src = build_source("s2_source", c_x[s2_train], c_y[s2_train], "copenhagen")
new = new_condition(src, s2_new, c_x, c_y)
PER_GROUP = [k for k in S2["per_group_grid"] if k not in (0, 10)]            # 1, 2, 4: these depend on the order
assert S2["per_group_grid"] == [0, 1, 2, 4, 10]
s2_reps = [r for r in S2["repetitions"] if r["order"] < S2_ORDERS]
if SMOKE:
    s2_reps = [r for r in S2["repetitions"] if r["choice"] == 0][:SMOKE_REPS]
# The smoke test runs all three values of every setting here, so that its timing is an upper bound for whichever
# value the full pseudo-transfer chooses (Codex review 003); the full run uses the chosen values only.
GRIDS_RUN_S2 = GRIDS_ALL if SMOKE else GRIDS_S2
pl, el, ids, full_by_choice, full_rows = [], [], [], {}, {}
for rep in s2_reps:
    pool_all = sorted(int(r) for g in rep["pool_groups"] for r in rep["pool_order"][g])
    assert set(pool_all) | set(rep["eval"]) == set(s2_new.tolist()) and not (set(pool_all) & set(rep["eval"]))
    assert not (set(c_group[pool_all].tolist()) & set(c_group[rep["eval"]].tolist())), "an evaluation group is in the update pool"
    c = rep["choice"]
    if c not in full_by_choice:
        # fits that do not depend on the order (no update; all 40 pool tablets): once per choice of pool groups,
        # update tablets in sorted row order, one fit seed per choice (plan section 3)
        full_by_choice[c], meta = run_part("s2", f"choice{c:02d}_full", src, new, {len(pool_all): pool_all}, rep["eval"], GRIDS_RUN_S2, True, c)
        full_rows[c] = (pool_all, rep["eval"])
        SECONDS.append(dict(unit="s2", part=f"choice{c:02d}_full", **{k: round(v, 2) for k, v in meta["seconds"].items()}, total=round(meta["seconds_total"], 2)))
    assert full_rows[c] == (pool_all, rep["eval"]), "orders of one choice must share the pool tablets and the evaluation order"
    upd_by_n = {}
    for k in PER_GROUP:
        upd_by_n[4 * k] = group_updates(rep, k)
    assert all(set(upd_by_n[a]) < set(upd_by_n[b]) for a, b in ((4, 8), (8, 16))) and set(upd_by_n[16]) < set(pool_all), "update sets must be nested"
    preds, meta = run_part("s2", f"rep{rep['rep']:03d}", src, new, upd_by_n, rep["eval"], GRIDS_RUN_S2, False, rep["rep"])
    SECONDS.append(dict(unit="s2", part=f"rep{rep['rep']:03d}", **{k: round(v, 2) for k, v in meta["seconds"].items()}, total=round(meta["seconds_total"], 2)))
    pl.append({**full_by_choice[c], **preds}); el.append(rep["eval"]); ids.append(rep["rep"])
    if len(ids) % 20 == 0:
        print("s2 repetitions done", len(ids), "of", len(s2_reps), "| minutes", round((time.time() - T0) / 60, 1), flush=True)
assemble("s2", ids, pl, el, c_y, expected_labels(GRIDS_RUN_S2, [4, 8, 16, 40], True), copenhagen_extra(s2_train))

In [ ]:
# 3. Scenario I: instrument change, both directions, all three values of every update setting.
# The pairing of the two instruments' rows is never used (deviation 1): source rows are training-part rows of the
# source instrument; update and evaluation rows are test-part rows of the new instrument.
s1_train = np.array(S1["train"])
s1_test = np.flatnonzero(A["s_part"] == "test")
assert len(s1_train) == 195 and len(s1_test) == 460 and not (set(s1_train.tolist()) & set(s1_test.tolist()))
assert S1["grid"] == [0, 5, 10, 20, 40, 80]
BENCH = []
for unit, a, b in (("s1_12", "1", "2"), ("s1_21", "2", "1")):
    src = build_source(f"s1_inst{a}", A[f"s_x{a}"][s1_train], A["s_assay"][s1_train], "shootout")
    new = new_condition(src, s1_test, A[f"s_x{b}"], A["s_assay"])
    # benchmark of deviation 1: the source model on all 460 test-part tablets measured on the source instrument
    own, _ = predict_source(src, A[f"s_x{a}"][s1_test])
    for lab, p in own.items():
        BENCH += [dict(unit=unit, source_instrument=int(a), family_label=lab, row=int(r), tablet=str(S1["tablet"][r]), reference=float(A["s_assay"][r]),
                       prediction=float(np.float32(v))) for r, v in zip(s1_test, p)]      # stored precision, as the curves
    reps = S1["repetitions"][:(SMOKE_REPS if SMOKE else S1_REPS)]
    pl, el, ids = [], [], []
    for rep in reps:
        pool, ev = rep["pool_order"], rep["eval"]
        assert len(pool) == 100 and len(ev) == 360 and set(pool) | set(ev) == set(s1_test.tolist()) and not (set(pool) & set(ev))
        upd_by_n = {n: pool[:n] for n in S1["grid"] if n > 0}
        preds, meta = run_part(unit, f"rep{rep['rep']:03d}", src, new, upd_by_n, ev, GRIDS_ALL, True, rep["rep"])
        SECONDS.append(dict(unit=unit, part=f"rep{rep['rep']:03d}", **{k: round(v, 2) for k, v in meta["seconds"].items()}, total=round(meta["seconds_total"], 2)))
        pl.append(preds); el.append(ev); ids.append(rep["rep"])
        if len(ids) % 10 == 0:
            print(unit, "repetitions done", len(ids), "of", len(reps), "| minutes", round((time.time() - T0) / 60, 1), flush=True)
    assemble(unit, ids, pl, el, A["s_assay"], expected_labels(GRIDS_ALL, [n for n in S1["grid"] if n > 0], True))

In [ ]:
# Timing and the estimate for the cost rule (plan section 7): mean seconds per part, times the full design.
sec_df = pd.DataFrame(SECONDS).fillna(0.0)
sec_df["kind"] = np.where(sec_df["part"].str.contains("full"), "per choice", "per repetition")
per = sec_df.groupby(["unit", "kind"])["total"].mean()
FULL_DESIGN = {("pseudo_01", "per repetition"): 27, ("pseudo_10", "per repetition"): 8, ("s2", "per choice"): 81, ("s2", "per repetition"): 243,
               ("s1_12", "per repetition"): 200, ("s1_21", "per repetition"): 200}
assert set(per.index) == set(FULL_DESIGN)
est = {f"{u} ({k})": dict(mean_seconds=float(per[(u, k)]), parts_in_full_design=n, hours=float(per[(u, k)]) * n / 3600) for (u, k), n in FULL_DESIGN.items()}


def design_hours(s1_reps, s2_orders):
    """Hours for one of the cost rule's designs: all 35 pseudo-transfer parts and all 81 order-free parts are always kept."""
    return float((per[("pseudo_01", "per repetition")] * 27 + per[("pseudo_10", "per repetition")] * 8 + per[("s2", "per choice")] * 81
                  + per[("s2", "per repetition")] * 81 * s2_orders + (per[("s1_12", "per repetition")] + per[("s1_21", "per repetition")]) * s1_reps) / 3600)


timing_doc = dict(estimated_hours={"200 repetitions, 3 orders (the plan)": design_hours(200, 3), "100 repetitions, 2 orders": design_hours(100, 2),
                                   "100 repetitions, 1 order": design_hours(100, 1)},
                  by_part=est, parts_timed=int(len(sec_df)), smoke=SMOKE, elapsed_minutes_this_run=(time.time() - T0) / 60,
                  gpu_load_before_this_run=GPU_BEFORE,
                  note="Wall-clock on this PC from the part files (a resumed run reports the times of when each part was computed). Not included: "
                       "source-model set-up, no-update and benchmark predictions, file writing and assembly (see elapsed minutes). In the smoke test "
                       "Scenario II runs all three values of every update setting, so its share is an upper bound for any chosen value. "
                       "A GPU load well above zero before this run means the GPU was shared and the times are inflated.")
print(json.dumps(timing_doc, indent=1))

In [ ]:
# Write outputs: every file to a temporary name first; all are renamed after all were written; a completion marker
# with their hashes is written last. Notebook 04 rebuilds every table and figure from the prediction files.
summary_df = pd.DataFrame(SUMMARY)
M.require_finite(summary_df["rmse"].to_numpy(), "summary")
texts = {
    "p4_curves_summary.csv": summary_df.to_csv(index=False, lineterminator="\n"),
    "p4_pseudo_transfer.csv": pt_table.to_csv(index=False, lineterminator="\n"),
    "p4_bench_s1.csv": pd.DataFrame(BENCH).to_csv(index=False, lineterminator="\n"),
    "p4_curves_seconds.csv": sec_df.to_csv(index=False, lineterminator="\n"),
    "p4_curves_timing.json": json.dumps(timing_doc, sort_keys=True, indent=1),
    "p4_versions_03.csv": pd.DataFrame([dict(item=k, value=v) for k, v in dict(VERSIONS, S1_REPS=S1_REPS, S2_ORDERS=S2_ORDERS, fingerprint=FINGERPRINT).items()]).to_csv(index=False, lineterminator="\n"),
}
described = {
    "p4_curves_summary.csv": ("scores", "RMSE and bias per part of the design, repetition, family, CNN seed, update, setting and update size n, computed from the saved 32-bit predictions"),
    "p4_pseudo_transfer.csv": ("pseudo-transfer", "Pseudo-transfer inside the source scales: mean RMSE at n = 8 per direction, update method and setting (the table behind p4_update_settings.json)"),
    "p4_bench_s1.csv": ("Scenario I", "Scenario I benchmark of deviation 1: each source model's prediction for all 460 test-part tablets measured on the source instrument"),
    "p4_curves_seconds.csv": ("timing", "Wall-clock seconds per part file (repetition or choice of pool groups) and model family"),
    "p4_curves_timing.json": ("timing", "Estimated hours of the full design from the mean seconds per part (input of the cost rule)"),
    "p4_versions_03.csv": ("versions", "Package versions, GPU, design size and run fingerprint of the notebook 03 run"),
    "p4_update_settings.json": ("pseudo-transfer", "Update settings: the pseudo-transfer choice for Scenario II with its table and provenance, and the main (middle) settings of Scenario I"),
}
for unit, u in UNITS.items():
    name = f"p4_pred_{unit}.npz"
    np.savez_compressed(os.path.join(OUT, name[:-4] + ".tmp.npz"), **u)
    described[name] = ("predictions", f"Per-tablet predictions of part {unit}: pred[repetition, method label, evaluation tablet] (32-bit), labels made of family, update, setting and n, eval_rows (row numbers in the arrays), y_true, rep")
for name, text in texts.items():
    with open(os.path.join(OUT, name + ".tmp"), "w", encoding="utf-8", newline="\n") as f:
        f.write(text)
for name in texts:
    os.replace(os.path.join(OUT, name + ".tmp"), os.path.join(OUT, name))
for unit in UNITS:
    os.replace(os.path.join(OUT, f"p4_pred_{unit}.tmp.npz"), os.path.join(OUT, f"p4_pred_{unit}.npz"))
files = list(texts) + [f"p4_pred_{u}.npz" for u in UNITS] + ["p4_update_settings.json"]
if not SMOKE:
    rows = [(n, NOTEBOOK, described[n][0], described[n][1]) for n in files]
    rows.append(("p4_complete_03.json", NOTEBOOK, "outputs", "Completion marker of notebook 03: run fingerprint, design size and the hash of each of its output files"))
    man_path = os.path.join(OUT, "MANIFEST.md")
    HEADER = ["| file | notebook | cell | description |", "|---|---|---|---|"]
    with open(man_path, encoding="utf-8") as f:
        lines = f.read().splitlines()
    other = [l for l in lines if l.startswith("|") and l not in HEADER and f"| {NOTEBOOK} |" not in l]
    with open(man_path + ".tmp", "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(HEADER + other + [f"| {a} | {b} | {c} | {d} |" for a, b, c, d in rows]) + "\n")
    os.replace(man_path + ".tmp", man_path)
marker = dict(notebook=NOTEBOOK, fingerprint=FINGERPRINT, smoke=SMOKE, S1_REPS=S1_REPS, S2_ORDERS=S2_ORDERS,
              repetitions={u: int(len(v["rep"])) for u, v in UNITS.items()}, files={n: P.sha256_file(os.path.join(OUT, n)) for n in files})
with open(os.path.join(OUT, "p4_complete_03.json.tmp"), "w", encoding="utf-8", newline="\n") as f:
    f.write(json.dumps(marker, sort_keys=True, indent=1))
os.replace(os.path.join(OUT, "p4_complete_03.json.tmp"), os.path.join(OUT, "p4_complete_03.json"))
print("DONE", NOTEBOOK, "| SMOKE", SMOKE, "| minutes", round((time.time() - T0) / 60, 1))